# Rotary Positional Embeddings (RoPE)

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

RoPE codifica posição *rotacionando* os vetores de query e key em subespaços 2-D. O produto interno $\langle q, k\rangle$ passa a depender só da posição *relativa* $m - n$. Zero parâmetros aprendidos, suporta extrapolação e integra direto na self-attention.


## Formulação Matemática

Para cada par de canais $(2i, 2i+1)$ rotacionamos pelo ângulo $\theta_i\,m$ com $\theta_i = 10000^{-2i/d}$:

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1}\end{pmatrix} = \begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i\end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1}\end{pmatrix}$$

Daí $\langle R_m q, R_n k\rangle = \langle q, R_{m-n} k\rangle$, ou seja, só a *diferença* importa.


## Implementação


In [ ]:
import math, torch


In [ ]:
def rope_freqs(d, max_len, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    pos = torch.arange(max_len).float()
    freqs = torch.einsum('i,j->ij', pos, inv)   # (L, d/2)
    return torch.polar(torch.ones_like(freqs), freqs)  # complex (L, d/2)

def apply_rope(x, freqs):
    """x: (B, H, L, D) where D is even."""
    B, H, L, D = x.shape
    x_c = torch.view_as_complex(x.float().reshape(B, H, L, D // 2, 2))
    x_rot = x_c * freqs[:L].unsqueeze(0).unsqueeze(0)
    return torch.view_as_real(x_rot).reshape(B, H, L, D)


## Experimento


In [ ]:
torch.manual_seed(0)
B, H, L, D = 1, 1, 8, 16
q = torch.randn(B, H, L, D)
k = torch.randn(B, H, L, D)

freqs = rope_freqs(D, L)
qr, kr = apply_rope(q, freqs), apply_rope(k, freqs)

# Translation invariance: rotate both by 3 positions and the relative score should match
freqs_shift = rope_freqs(D, L + 3)
qr2 = apply_rope(q, freqs_shift[3:3+L])
kr2 = apply_rope(k, freqs_shift[3:3+L])

print('rel score same?', torch.allclose((qr @ kr.transpose(-2, -1)), (qr2 @ kr2.transpose(-2, -1)), atol=1e-5))


## Discussão

- RoPE é a posição padrão de fato em LLaMA, GPT-NeoX, Qwen, etc.
- Truques de extrapolação como *NTK-aware* scaling ou *YaRN* permitem ir além do contexto de treino.
- Custo praticamente zero: um tensor complexo pré-computado e uma multiplicação element-wise.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
